# Ariane — MaskPlace RL (PPO) on the hard macros

Trains the **original MaskPlace** PPO agent (`PPO2.py`, ResNet18 + WireMask/PositionMask) to place the **same 133 ariane hard macros** that the LLM/greedy study (`ariane-full-matrix.ipynb`) places. The RL agent learns *where* to put each macro (fixed MaskPlace topology order); the LLM-ordering method learns the *order* (greedy positions). Both score with the **identical `comp_res()` HPWL**, so the numbers are directly comparable.

**Comparison metric:** *best HPWL seen during training* (MaskPlace's own convention).

### Kaggle settings (do this first, right sidebar)
- **Accelerator = GPU** (T4 x2 or P100).
- **Internet = On** (ResNet18 downloads pretrained weights; the trainer falls back to random init if off).

### What to expect
- Each episode places 133 macros (~a few seconds/episode; Python WireMask + ResNet18 forward dominate).
- A first clearly-improving HPWL appears within ~30–90 min; a strong/plateaued HPWL typically needs **~4–10 h**.
- Free GPU = ~30 h/week, ~12 h/session. One run fits a session; best-so-far is saved continuously so you can stop anytime.
- **Cost = $0** (no Anthropic API here).

## Cell A — Setup (clone repo, install gym + protobuf, check GPU)
Needs `PPO2.py` pushed to the repo.

In [ ]:
import subprocess, sys, os, shutil, glob

REPO_URL  = "https://github.com/dennis5727/arianePlacement.git"
CLONE_DIR = "/kaggle/working/arianePlacement"
WORK_DIR  = os.path.join(CLONE_DIR, "maskplace")

if not os.path.exists(CLONE_DIR):
    try:
        subprocess.check_call(["git","clone","--depth","1",REPO_URL,CLONE_DIR]); print("cloned")
    except Exception as e: print("git clone failed:", e)
else:
    subprocess.call(["git","-C",CLONE_DIR,"pull","--ff-only"]); print("pulled latest")

if not os.path.exists(WORK_DIR):
    hits=glob.glob("/kaggle/input/**/place_db.py",recursive=True)
    assert hits,"no code via git or dataset"
    shutil.copytree(os.path.dirname(hits[0]),WORK_DIR); print("dataset fallback")

os.chdir(WORK_DIR); sys.path.insert(0,WORK_DIR); print("cwd:",os.getcwd())

# protobuf pin for the ariane netlist parser
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION","python")
try: subprocess.check_call([sys.executable,"-m","pip","install","-q","protobuf==3.20.3"])
except subprocess.CalledProcessError as e: print("protobuf pin failed:",e)

# REAL gym is required (PPO2.py uses gym.make + registration). The 4-tuple step
# API and env.seed() exist in <=0.25, so pin there; 0.26+ would break PPO2.py.
def _gym_ok():
    try:
        import gym; v=tuple(int(x) for x in gym.__version__.split('.')[:2]); return v<=(0,25)
    except Exception: return False
if not _gym_ok():
    for spec in ["gym==0.25.2","gym==0.23.1","gym==0.21.0"]:
        try:
            subprocess.check_call([sys.executable,"-m","pip","install","-q",spec])
            import importlib, gym; importlib.reload(gym)
            if _gym_ok(): break
        except subprocess.CalledProcessError as e: print(spec,"failed:",e)
import gym; print("gym:", gym.__version__)

import torch, torchvision
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__,
      "| CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
if not torch.cuda.is_available():
    print("WARNING: no GPU — set Accelerator=GPU in the sidebar (training is very slow on CPU).")

need=["PPO2.py","place_db.py","place_env/place_env.py","comp_res.py","prim.py",
      "place_db_proto.py","ariane/netlist.pb.txt"]
miss=[f for f in need if not os.path.exists(f)]
assert not miss, f"MISSING (push to repo): {miss}"
print("files OK\n=== SETUP COMPLETE ===")

## Cell B — Sanity check (ariane = 133 hard macros, canvas 357)

In [ ]:
from place_db import PlaceDB
placedb = PlaceDB("ariane")
hard = sum(1 for n in placedb.node_info if placedb.node_info[n].get("is_hard"))
print("Nodes", placedb.node_cnt, "| Nets", placedb.net_cnt,
      "| Canvas", placedb.max_height, "| Hard macros", hard)
assert placedb.max_height == 357
assert hard == 133, "expected 133 hard macros for ariane, got {}".format(hard)
# ariane has 932 nodes but only 133 are HARD macros; the greedy/LLM study places
# only those 133, so we run RL with --hard_only --pnm 133 to match exactly.
print("PNM = {} hard macros (same set the LLM/greedy study places)".format(hard))

## Cell C — Smoke test (~300 episodes, validates the pipeline end-to-end)
Confirms episodes complete, HPWL prints and decreases, and the best-HPWL CSV / model / figure are written to `OUTDIR`. Expect ~15–45 min. Watch for `*** NEW BEST HPWL ... ***` lines.

In [ ]:
import os, subprocess, sys
OUTDIR = "/kaggle/working/rl_out"
os.makedirs(OUTDIR, exist_ok=True)

def run_ppo(max_episodes, pnm=133, seed=42, lr=2.5e-3, extra=None):
    # --hard_only places ONLY the 133 is_hard macros (topology order) = the same
    # set/order the greedy "strong (topo)" baseline uses, so HPWL is comparable.
    cmd = [sys.executable, "-u", "PPO2.py",
           "--benchmark", "ariane", "--hard_only", "--pnm", str(pnm),
           "--seed", str(seed), "--lr", str(lr), "--max_episodes", str(max_episodes),
           "--outdir", OUTDIR, "--save_fig"]
    if extra: cmd += extra
    print("RUN:", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=os.getcwd(), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        p.terminate(); print("\n[interrupted — best-so-far is already saved in OUTDIR]")
    p.wait(); return p.returncode

run_ppo(max_episodes=300)

## Cell D — Full training run
Same command, large episode cap. It keeps saving best-so-far HPWL / model / placement to `OUTDIR`. **Stop the cell** (interrupt) when the session nears the 12 h limit or HPWL plateaus — the best is already on disk. Re-run with a different `--seed` for another sample.

In [ ]:
run_ppo(max_episodes=100000)

## Cell E — Compare: MaskPlace RL vs greedy strong baseline
Reads the best-HPWL CSV and computes the project's greedy **strong (topology)** baseline live, in the same `comp_res` units. Add your LLM-ordering best from `ariane-full-matrix.ipynb` for the full comparison.

In [ ]:
import csv, glob, os

# --- MaskPlace RL: best HPWL seen during training ---
rl_best = None
csvs = sorted(glob.glob(os.path.join("/kaggle/working/rl_out", "best_hpwl_ariane_*.csv")))
for path in csvs:
    with open(path) as f:
        for row in csv.DictReader(f):
            h = float(row["hpwl"])
            if rl_best is None or h < rl_best["hpwl"]:
                rl_best = {"hpwl": h, "cost": float(row["cost"]),
                           "episode": int(row["episode"]), "file": os.path.basename(path)}
print("MaskPlace RL best:", rl_best)

# --- Project greedy STRONG (topology) baseline, same comp_res units ---
from strong_search import hard_macro_names, connectivity_order, greedy_hpwl
hard = hard_macro_names(placedb)
_, greedy_strong_hpwl, _ = greedy_hpwl(placedb, connectivity_order(placedb, hard), grid=224)
print("greedy strong (topo) HPWL:", greedy_strong_hpwl)

# --- paste your LLM-ordering best from ariane-full-matrix.ipynb here ---
llm_ordering_best = None  # e.g. 8.0e4

print("\n" + "="*56)
print(f"{'method':<34}{'HPWL':>14}{'vs greedy':>8}")
print("-"*56)
def row(name, h):
    if h is None: print(f"{name:<34}{'(fill in)':>14}"); return
    d = 100.0*(greedy_strong_hpwl - h)/greedy_strong_hpwl
    print(f"{name:<34}{h:>14.4e}{d:>+7.1f}%")
row("greedy strong (topo) baseline", greedy_strong_hpwl)
row("LLM ordering (best)", llm_ordering_best)
row("MaskPlace RL (best during training)", rl_best["hpwl"] if rl_best else None)
print("="*56)
print("(positive vs greedy = lower HPWL = better)")